# Support IN operator in CQL2 filters

RSPY-554

The goal of this story is to add support of the IN operator.

**Applicable documents**

  * https://docs.ogc.org/is/21-065r2/21-065r2.html#advanced-comparison-operators


The searches below cover RSPY-1216 (constellation/platform pairs) and RSPY-1217 (CQL2 IN filtering), using GET and POST.
Assertions check the expected IDs and platforms against the local station mock data.


In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

In [ ]:
# Other imports
from collections import defaultdict
import ipywidgets as widgets

In [ ]:
search_all = widgets.Checkbox(
    value=False,
    description="Run a search on all products first ?",
    indent=False,
)
display(search_all)

## Auxip search

In [ ]:
# (Optional) get 20 first products, print their properties
if search_all.value:
    results = auxip_client.search(method="GET", limit=20, max_items=50)
    display(results)
    ids = []
    platforms = defaultdict(int)
    ptypes = defaultdict(int)
    for feature in results:
        ids.append(feature.id)
        platforms[feature.properties["platform"]] += 1
    
    print("Search everything:")
    display(ids)
    display(platforms)

print("\nSearch on several ids with filter:")
ids = [
    "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000", 
    "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000"
]
id_filter = ','.join([f"'{id}'" for id in ids])
id_filter = f"id in ({id_filter})"
display(id_filter)
results = auxip_client.search(method="GET", stac_filter=id_filter)
display (results)
# Check results
assert len(results) == 2
assert {feature.id for feature in results} == set(ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"

print("\nSearch on several platforms with filter:")
platforms = ["sentinel-1a", "sentinel-2b"]
platform_filter = ','.join([f"'{platform}'" for platform in platforms])
platform_filter = f"platform in ({platform_filter})"
display(platform_filter)
results = auxip_client.search(method="GET", stac_filter=platform_filter)
display (results)
# Check results
for feature in results:
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

# The selected AUXIP products belong to S1A and S2A; only S1A matches this filter.
expected_ids = [ids[0]]

print("\nSearch with all filters:")
all_filter = " and ".join([id_filter, platform_filter])
display(all_filter)
results = auxip_client.search(method="GET", stac_filter=all_filter)
display (results)
# Check results
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

print("\nSearch with POST method:")
post_filter = {
  "op": "and",
  "args": [
      {"op": "in", "args": [{"property": "id" }, ids]},
      {"op": "in", "args": [{"property": "platform" }, platforms]},
  ]
}
results = auxip_client.search(method="POST", stac_filter=post_filter)
display (results)
# Check results
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

### RSPY-1217: exact names and attribute lists

These examples use the local station mock data. Each GET and POST checks the complete expected set of IDs.


In [ ]:
# Exact station filenames; STAC IDs omit the archive extension.
expected_ids = [
    "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000",
    "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000"
]

print("Search with GET method:")
stac_filter = "Name IN ('S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml','S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000.TGZ')"
display(stac_filter)
results = auxip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "Name"
        },
        [
            "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml",
            "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000.TGZ"
        ]
    ]
}
display(post_filter)
results = auxip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)


In [ ]:
# Multiple product types in the station Attributes collection.
expected_ids = [
    "S1A_AUX_PP2_V20200516T080000_G20200516T080000",
    "S1A_OPER_AUX_OBMEMC_PDMC_20210924T000000",
    "S1A_OPER_AUX_PREORB_OPOD_20200822T062732_V20200822T053140_20200822T120640",
    "S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732",
    "S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814",
    "S2__OPER_AUX_ECMWFD_PDMC_20190216T120000_V20190217T090000_20190217T210000",
    "S2__OPER_AUX_ECMWFD_PDMC_20200216T120000_V20190217T090000_20190217T210000",
    "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000",
    "S3B_AX___FRO_AX_20251005T000000_20251015T000000_20251010T143722___________________EUM_O_AL_001.SEN3"
]

print("Search with GET method:")
stac_filter = "product:type IN ('OPER_AUX_ECMWFD_PDMC','AX___FRO_AX')"
display(stac_filter)
results = auxip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["product:type"] in ['OPER_AUX_ECMWFD_PDMC', 'AX___FRO_AX']

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "product:type"
        },
        [
            "OPER_AUX_ECMWFD_PDMC",
            "AX___FRO_AX"
        ]
    ]
}
display(post_filter)
results = auxip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["product:type"] in ['OPER_AUX_ECMWFD_PDMC', 'AX___FRO_AX']


In [ ]:
# A singleton IN list still applies the attribute filter.
expected_ids = [
    "S1A_AUX_PP2_V20200516T080000_G20200516T080000",
    "S1A_OPER_AUX_OBMEMC_PDMC_20210924T000000",
    "S1A_OPER_AUX_PREORB_OPOD_20200822T062732_V20200822T053140_20200822T120640",
    "S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732",
    "S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814",
    "S2__OPER_AUX_ECMWFD_PDMC_20190216T120000_V20190217T090000_20190217T210000",
    "S2__OPER_AUX_ECMWFD_PDMC_20200216T120000_V20190217T090000_20190217T210000"
]

print("Search with GET method:")
stac_filter = "product:type IN ('OPER_AUX_ECMWFD_PDMC')"
display(stac_filter)
results = auxip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "product:type"
        },
        [
            "OPER_AUX_ECMWFD_PDMC"
        ]
    ]
}
display(post_filter)
results = auxip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)


In [ ]:
# An unmatched ID must not return the unfiltered catalog.
expected_ids = []

print("Search with GET method:")
stac_filter = "id IN ('ABSENT_RSPY1217_AUXIP')"
display(stac_filter)
results = auxip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "id"
        },
        [
            "ABSENT_RSPY1217_AUXIP"
        ]
    ]
}
display(post_filter)
results = auxip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)


In [ ]:
# RSPY-1216: preserve constellation/platform pairs across missions.
expected_ids = [
    "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000",
    "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000"
]

print("Search with GET method:")
stac_filter = "constellation IN ('sentinel-1','sentinel-2') AND platform IN ('sentinel-1a','sentinel-2a') AND id IN ('S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000','S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000')"
display(stac_filter)
results = auxip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
assert {(feature.properties["constellation"], feature.properties["platform"]) for feature in results} == {('sentinel-1', 'sentinel-1a'), ('sentinel-2', 'sentinel-2a')}

print("\nSearch with POST method:")
post_filter = {
    "op": "and",
    "args": [
        {
            "op": "in",
            "args": [
                {
                    "property": "constellation"
                },
                [
                    "sentinel-1",
                    "sentinel-2"
                ]
            ]
        },
        {
            "op": "in",
            "args": [
                {
                    "property": "platform"
                },
                [
                    "sentinel-1a",
                    "sentinel-2a"
                ]
            ]
        },
        {
            "op": "in",
            "args": [
                {
                    "property": "id"
                },
                [
                    "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000",
                    "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000"
                ]
            ]
        }
    ]
}
display(post_filter)
results = auxip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
assert {(feature.properties["constellation"], feature.properties["platform"]) for feature in results} == {('sentinel-1', 'sentinel-1a'), ('sentinel-2', 'sentinel-2a')}


## Cadip search

In [ ]:
# (Optional) get 20 first products, print their properties
if search_all.value:
    results = cadip_client.search(method="GET", limit=20, max_items=50)
    display(results)
    ids = []
    platforms = defaultdict(int)
    ptypes = defaultdict(int)
    for feature in results:
        ids.append(feature.id)
        platforms[feature.properties["platform"]] += 1
    
    print("Search everything:")
    display(ids)
    display(platforms)

print("\nSearch on several ids with filter:")
ids = [
    "S1A_20231120061537234567", 
    "S2B_20231117033237234567"
]
id_filter = ','.join([f"'{id}'" for id in ids])
id_filter = f"id in ({id_filter})"
display(id_filter)
results = cadip_client.search(method="GET", stac_filter=id_filter)
display (results)
# Check results
assert len(results) == 2
assert {feature.id for feature in results} == set(ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"

print("\nSearch on several platforms with filter:")
platforms = ["sentinel-1a", "sentinel-2b"]
platform_filter = ','.join([f"'{platform}'" for platform in platforms])
platform_filter = f"platform in ({platform_filter})"
display(platform_filter)
results = cadip_client.search(method="GET", stac_filter=platform_filter)
display (results)
# Check results
for feature in results:
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

expected_ids = ids

print("\nSearch with all filters:")
all_filter = " and ".join([id_filter, platform_filter])
display(all_filter)
results = cadip_client.search(method="GET", stac_filter=all_filter)
display (results)
# Check results
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

print("\nSearch with POST method:")
post_filter = {
  "op": "and",
  "args": [
      {"op": "in", "args": [{"property": "id" }, ids]},
      {"op": "in", "args": [{"property": "platform" }, platforms]},
  ]
}
results = cadip_client.search(method="POST", stac_filter=post_filter)
display (results)
# Check results
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

### RSPY-1217: antenna lists

The original antenna case, a singleton and an unmatched value are checked with GET and POST.


In [ ]:
# Original bug: filter sessions by an antenna list.
expected_ids = [
    "S1A_20200105072204051312",
    "S1A_20210410031928012345",
    "S1A_20220715090550123456",
    "S1A_20231120061537234567",
    "S2B_20231117033237234567",
    "S3A_20250109134406046340",
    "S3B_20251010143722593812"
]

print("Search with GET method:")
stac_filter = "cadip:antenna_id IN ('MSP21','other')"
display(stac_filter)
results = cadip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["cadip:antenna_id"] in ['MSP21', 'other']

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "cadip:antenna_id"
        },
        [
            "MSP21",
            "other"
        ]
    ]
}
display(post_filter)
results = cadip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["cadip:antenna_id"] in ['MSP21', 'other']


In [ ]:
# A singleton antenna list.
expected_ids = [
    "S1A_20200105072204051312",
    "S1A_20210410031928012345",
    "S1A_20220715090550123456",
    "S1A_20231120061537234567",
    "S2B_20231117033237234567",
    "S3A_20250109134406046340",
    "S3B_20251010143722593812"
]

print("Search with GET method:")
stac_filter = "cadip:antenna_id IN ('MSP21')"
display(stac_filter)
results = cadip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["cadip:antenna_id"] in ['MSP21']

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "cadip:antenna_id"
        },
        [
            "MSP21"
        ]
    ]
}
display(post_filter)
results = cadip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["cadip:antenna_id"] in ['MSP21']


In [ ]:
# An unmatched antenna must return no sessions.
expected_ids = []

print("Search with GET method:")
stac_filter = "cadip:antenna_id IN ('ABSENT_RSPY1217_CADIP')"
display(stac_filter)
results = cadip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["cadip:antenna_id"] in ['ABSENT_RSPY1217_CADIP']

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "cadip:antenna_id"
        },
        [
            "ABSENT_RSPY1217_CADIP"
        ]
    ]
}
display(post_filter)
results = cadip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["cadip:antenna_id"] in ['ABSENT_RSPY1217_CADIP']


## Prip search

In [ ]:
# (Optional) get 20 first products, print their properties
if search_all.value:
    results = prip_client.search(method="GET", limit=20, max_items=50)
    display(results)
    ids = []
    platforms = defaultdict(int)
    ptypes = defaultdict(int)
    for feature in results:
        ids.append(feature.id)
        platforms[feature.properties["platform"]] += 1
    
    print("Search everything:")
    display(ids)
    display(platforms)

print("\nSearch on several ids with filter:")
ids = [
    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE",
    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12",
]
id_filter = ','.join([f"'{id}'" for id in ids])
id_filter = f"id in ({id_filter})"
display(id_filter)
results = prip_client.search(method="GET", stac_filter=id_filter)
display (results)
# Check results
assert len(results) == 2
assert {feature.id for feature in results} == set(ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"

print("\nSearch on several platforms with filter:")
platforms = ["sentinel-1a", "sentinel-2b"]
platform_filter = ','.join([f"'{platform}'" for platform in platforms])
platform_filter = f"platform in ({platform_filter})"
display(platform_filter)
results = prip_client.search(method="GET", stac_filter=platform_filter)
display (results)
# Check results
for feature in results:
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms} ({feature.id})"

expected_ids = ids

print("\nSearch with all filters:")
all_filter = " and ".join([id_filter, platform_filter])
display(all_filter)
results = prip_client.search(method="GET", stac_filter=all_filter)
display (results)
# Check results
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

print("\nSearch with POST method:")
post_filter = {
  "op": "and",
  "args": [
      {"op": "in", "args": [{"property": "id" }, ids]},
      {"op": "in", "args": [{"property": "platform" }, platforms]},
  ]
}
results = prip_client.search(method="POST", stac_filter=post_filter)
display (results)
# Check results
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert (ret := feature.id) in ids, f"{ret} not in {ids}"
    assert (ret := feature.properties["platform"]) in platforms, f"{ret} not in {platforms}"

### RSPY-1217: enum, numeric and datetime lists

The filters exercise the typed OData mappings. Integer and integral-float inputs must select the same products.


In [ ]:
# ProductionType enum members through processing:lineage.
expected_ids = [
    "S2B_OPER_MSI_L0__DS_2BPS_20250801T074015_S20250801T070620_N05.11",
    "S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11",
    "S2B_OPER_MSI_L2A_TL_2BPS_20250801T074829_A043890_T47XMH_N05.11",
    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12"
]

print("Search with GET method:")
stac_filter = "processing:lineage IN ('systematic_production','on-demand default')"
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["processing:lineage"] in ['systematic_production', 'on-demand default']

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "processing:lineage"
        },
        [
            "systematic_production",
            "on-demand default"
        ]
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["processing:lineage"] in ['systematic_production', 'on-demand default']


In [ ]:
# Integral orbit values: [43829, 47051].
expected_ids = [
    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE",
    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12"
]

print("Search with GET method:")
stac_filter = 'sat:absolute_orbit IN (43829,47051)'
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["sat:absolute_orbit"] in [43829, 47051]

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "sat:absolute_orbit"
        },
        [
            43829,
            47051
        ]
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["sat:absolute_orbit"] in [43829, 47051]


In [ ]:
# Integral orbit values: [43829.0, 47051.0].
expected_ids = [
    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE",
    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12"
]

print("Search with GET method:")
stac_filter = 'sat:absolute_orbit IN (43829.0,47051.0)'
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["sat:absolute_orbit"] in [43829, 47051]

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "sat:absolute_orbit"
        },
        [
            43829.0,
            47051.0
        ]
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
for feature in results:
    assert feature.properties["sat:absolute_orbit"] in [43829, 47051]


In [ ]:
# Exact processing instants in a list.
expected_ids = [
    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE",
    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12"
]

print("Search with GET method:")
stac_filter = "processing:datetime IN (TIMESTAMP('2022-06-26T06:11:22.535Z'),TIMESTAMP('2026-03-10T13:58:37Z'))"
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "processing:datetime"
        },
        [
            {
                "timestamp": "2022-06-26T06:11:22.535Z"
            },
            {
                "timestamp": "2026-03-10T13:58:37Z"
            }
        ]
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)


In [ ]:
# Exact processing instants in a singleton list.
expected_ids = [
    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE"
]

print("Search with GET method:")
stac_filter = "processing:datetime IN (TIMESTAMP('2022-06-26T06:11:22.535Z'))"
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "processing:datetime"
        },
        [
            {
                "timestamp": "2022-06-26T06:11:22.535Z"
            }
        ]
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)


In [ ]:
# An unmatched ID must not return the unfiltered catalog.
expected_ids = []

print("Search with GET method:")
stac_filter = "id IN ('ABSENT_RSPY1217_PRIP')"
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)

print("\nSearch with POST method:")
post_filter = {
    "op": "in",
    "args": [
        {
            "property": "id"
        },
        [
            "ABSENT_RSPY1217_PRIP"
        ]
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)


In [ ]:
# RSPY-1216: preserve constellation/platform pairs across missions.
expected_ids = [
    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE",
    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12"
]

print("Search with GET method:")
stac_filter = "constellation IN ('sentinel-1','sentinel-2') AND platform IN ('sentinel-1a','sentinel-2b') AND id IN ('S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE','S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12')"
display(stac_filter)
results = prip_client.search(method="GET", stac_filter=stac_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
assert {(feature.properties["constellation"], feature.properties["platform"]) for feature in results} == {('sentinel-1', 'sentinel-1a'), ('sentinel-2', 'sentinel-2b')}

print("\nSearch with POST method:")
post_filter = {
    "op": "and",
    "args": [
        {
            "op": "in",
            "args": [
                {
                    "property": "constellation"
                },
                [
                    "sentinel-1",
                    "sentinel-2"
                ]
            ]
        },
        {
            "op": "in",
            "args": [
                {
                    "property": "platform"
                },
                [
                    "sentinel-1a",
                    "sentinel-2b"
                ]
            ]
        },
        {
            "op": "in",
            "args": [
                {
                    "property": "id"
                },
                [
                    "S1A_IW_RAW__0NSH_20220626T050533_20220626T051038_043829_053B7F_203C.SAFE",
                    "S2B_OPER_MSI_L2A_TL_2BPS_20260310T135837_A047051_T35MMN_N05.12"
                ]
            ]
        }
    ]
}
display(post_filter)
results = prip_client.search(method="POST", stac_filter=post_filter, limit=1000, max_items=1000)
display(results)
assert len(results) == len(expected_ids)
assert {feature.id for feature in results} == set(expected_ids)
assert {(feature.properties["constellation"], feature.properties["platform"]) for feature in results} == {('sentinel-1', 'sentinel-1a'), ('sentinel-2', 'sentinel-2b')}


## Auxip staging using Prefect flows and CQL2 filters

In [ ]:
from resources.widget_utils import *
from rs_workflows.aux_flow import aux_staging
from resources.dask_clusters.dask_main_env import *

In [ ]:
await init_dask_cluster_staging()

In [ ]:
# Choose prefect deployment method
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Choose log level
prefect_logging_level_radio

In [ ]:
# Deploy the Prefect flows
s3_code_folder = f"users/{OWNER_ID}/code"
dpr_processing_deploy, aux_staging_deploy, cadip_staging_deploy = await deploy_prefect(
    deploy_file="../sprint38/dpr_processing_flow/dpr_processing_flow.yaml", 
    s3_code_folder=s3_code_folder, 
    work_pool_name=os.environ["PREFECT_WORK_POOL_SANDBOX"]
)

In [ ]:
# Create test collection
create_test_collection("CQL2_IN_OPERATOR_AUXIP")
  
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
    "logging_level": prefect_logging_level_radio.value,
  },
}

In [ ]:
# Auxip staging
ids = [
    "S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000", 
    "S2__OPER_AUX_ECMWFD_PDMC_20230216T120000_V20190217T090000_20190217T210000"
]
id_filter = ','.join([f"'{id}'" for id in ids])
id_filter = f"id in ({id_filter})"
cql2_filter = {"filter": id_filter}
params = {
    **flow_env_args,
    "cql2_filter": cql2_filter,
    "catalog_collection_identifier": "CQL2_IN_OPERATOR_AUXIP",
}
await run_prefect(
    deploy_name=aux_staging_deploy, 
    py_func=aux_staging, 
    params=params
)

# Check staged items in the catalog
staged = catalog_client.search(
    owner_id=OWNER_ID, 
    collections=["CQL2_IN_OPERATOR_AUXIP"]
).items
display(staged)
assert len(staged) == 2
for item in staged:
    assert (ret := item.id) in ids, f"{ret} not in {ids}"